<a href="https://colab.research.google.com/github/guilhermefrrr/ucl2425/blob/main/python-notebooks/extracting-fixtures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Presentation

This Python code is part of a football data analysis project, focusing on extracting and processing individual player statistics. It utilizes the pandas library to retrieve data from the FBref website, parsing HTML tables from match pages and organizing them into a structured format.

The project comprises three main steps using Python:

1. **Extracting stats:** This step is the core of the project. It extracts a wide range of player statistics, encompassing passing, shooting, and defensive actions. This data is then consolidated into a single comprehensive DataFrame that captures all individual player performances throughout the competition.
2. **Field Tilt:** Utilizing the DataFrame created in the previous step, this part of the project calculates the Field Tilt for each club across the competition. Field Tilt is a metric that quantifies a team's territorial dominance during a match, often estimated by comparing possession percentages or the location of attacking actions. A higher Field Tilt indicates greater control over the game.
3. **Extracting fixtures (this code):** This step focuses on gathering general information about each match in the competition. It extracts details such as the date and location of each game, providing context for the individual player statistics and Field Tilt calculations.

All the data collected across these three steps is then integrated into a Power BI report. This report leverages the extracted statistics and calculated metrics to provide interactive dashboards and visualizations, enabling in-depth analysis of player and team performance throughout the competition.

The code in this file (written below) iterates through a list of match URLs, extracting the relevant information and introducing a 10-second delay between requests to the FBref server to avoid overloading and potential blocking. Maintaining this delay function is crucial for the smooth operation of the process.

*The project is intended for educational and personal use only. FBref has its own terms of service and usage policies that should be respected. Please use this project responsibly and ethically.*

## Libraries

In [ ]:
import pandas as pd
from google.colab import drive

In [ ]:
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns

In [ ]:
df = pd.read_html('https://fbref.com/en/comps/8/schedule/Champions-League-Scores-and-Fixtures', attrs={"id": "sched_all"})[0]

In [ ]:
# Getting data from other file of this project to compare squads names
df1 = pd.read_csv('https://raw.githubusercontent.com/guilhermefrrr/ucl2425/refs/heads/main/complete-data/statsucl2425.csv')

## Data processing

In [ ]:
# Cleans the 'Home' and 'Away' columns by removing country abbreviations using regex
df['Home'] = df['Home'].str.replace(r'\s[a-zA-Z]{2,3}$', '', regex=True, case=False)
df['Away'] = df['Away'].str.replace(r'^[a-zA-Z]{2,3}\s', '', regex=True, case=False)

In [ ]:
# Drops the 'Match Report' and 'Notes' columns
df = df.drop(columns=['Match Report'])
df = df.drop(columns=['Notes'])

In [ ]:
# Removes rows with missing values in the 'Score' column
df = df.dropna(subset=['Score'])

In [ ]:
# Renames the 'Wk' column to 'MD'
df = df.rename(columns={'Wk': 'MD'})

# Removes rows where the the header values are replicated
# The code below checks especifically the rows where the 'Score' column contains the string 'Score'
df = df[df['Score'] != 'Score']

In [ ]:
# Converts the 'MD' column to numeric data type
df['MD'] = pd.to_numeric(df['MD'], errors='coerce')

In [ ]:
# Inserts a new column 'Match_ID' with a sequence of numbers
df.insert(df.columns.get_loc('MD') + 1, 'Match_ID', range(1, len(df) + 1))

In [ ]:
# Split the 'Score' column
df[['Score_Home', 'Score_Away']] = df['Score'].str.split('–', expand=True)

In [ ]:
# Move the 'Score_Home' and 'Score_Away' columns. Then, removes the 'Score' column
score_index = df.columns.get_loc('Score')
df.insert(score_index, 'Score_Home', df.pop('Score_Home'))
df.insert(score_index + 1, 'Score_Away', df.pop('Score_Away'))
df = df.drop(columns=['Score'])

In [ ]:
# Renames 'xG' to 'xG_Home' and 'xG.1' to 'xG_Away'
df = df.rename(columns={'xG': 'xG_Home', 'xG.1': 'xG_Away'})

In [ ]:
df.head()

,Round,MD,Match_ID,Day,Date,Time,Home,xG_Home,Score_Home,Score_Away,xG_Away,Away,Attendance,Venue,Referee
0,League phase,1.0,1,Tue,2024-09-17,18:45,Young Boys,0.6,0,3,2.0,Aston Villa,31500,Stadion Wankdorf,Georgi Kabakov
1,League phase,1.0,2,Tue,2024-09-17,18:45,Juventus,2.9,3,1,1.0,PSV Eindhoven,40417,Allianz Stadium,Alejandro Hernández
2,League phase,1.0,3,Tue,2024-09-17,20:00,Sporting CP,1.6,2,0,0.4,Lille,40024,Estádio José Alvalade,Donatas Rumšas
3,League phase,1.0,4,Tue,2024-09-17,21:00,Real Madrid,2.6,3,1,1.9,Stuttgart,71288,Estadio Santiago Bernabéu,Halil Umut Meler
4,League phase,1.0,5,Tue,2024-09-17,21:00,Milan,0.6,1,3,3.1,Liverpool,59826,Stadio Giuseppe Meazza,Espen Eskås


In [ ]:
# Gets unique squad names from the 'Home' and 'Away' columns in both df and df1
squads_df = pd.unique(df[['Home', 'Away']].values.ravel('K'))
squads_df1 = df1['Squad'].unique()

# Squads in df and out of df1
not_in_df1 = list(set(squads_df) - set(squads_df1))

# Squads in df1 and out of df
not_in_df = list(set(squads_df1) - set(squads_df))

print("Squads in df and out of df1:", not_in_df1)
print("Squads in df1 and out of df:", not_in_df)

Squads in df and out of df1: []
Squads in df1 and out of df: []


In [ ]:
squads_mapping = {
    'Shakhtar': 'Shakhtar Donetsk',
    'Leverkusen': 'Bayer Leverkusen',
    'Paris S-G': 'Paris Saint-Germain',
    'Red Star': 'Red Star Belgrade',
    'RB Salzburg': 'Red Bull Salzburg',
    'Inter': 'Internazionale'
}

In [ ]:
df['Home'] = df['Home'].replace(squads_mapping)
df['Away'] = df['Away'].replace(squads_mapping)

In [ ]:
df.head()

,Round,MD,Match_ID,Day,Date,Time,Home,xG_Home,Score_Home,Score_Away,xG_Away,Away,Attendance,Venue,Referee
0,League phase,1.0,1,Tue,2024-09-17,18:45,Young Boys,0.6,0,3,2.0,Aston Villa,31500,Stadion Wankdorf,Georgi Kabakov
1,League phase,1.0,2,Tue,2024-09-17,18:45,Juventus,2.9,3,1,1.0,PSV Eindhoven,40417,Allianz Stadium,Alejandro Hernández
2,League phase,1.0,3,Tue,2024-09-17,20:00,Sporting CP,1.6,2,0,0.4,Lille,40024,Estádio José Alvalade,Donatas Rumšas
3,League phase,1.0,4,Tue,2024-09-17,21:00,Real Madrid,2.6,3,1,1.9,Stuttgart,71288,Estadio Santiago Bernabéu,Halil Umut Meler
4,League phase,1.0,5,Tue,2024-09-17,21:00,Milan,0.6,1,3,3.1,Liverpool,59826,Stadio Giuseppe Meazza,Espen Eskås


In [ ]:
squads_df = pd.unique(df[['Home', 'Away']].values.ravel('K'))
squads_df1 = df1['Squad'].unique()

# Squads in df and out of df1
not_in_df1 = list(set(squads_df) - set(squads_df1))

# Squads in df1 and out of df
not_in_df = list(set(squads_df1) - set(squads_df))

print("Squads in df and out of df1:", not_in_df1)
print("Squads in df1 and out of df:", not_in_df)

Squads in df and out of df1: ['Paris Saint-Germain', 'Internazionale', 'Bayer Leverkusen', 'Red Bull Salzburg', 'Red Star Belgrade', 'Shakhtar Donetsk']
Squads in df1 and out of df: ['Leverkusen', 'Shakhtar', 'RB Salzburg', 'Inter', 'Paris S-G', 'Red Star']


In [ ]:
file_path = '/content/drive/MyDrive/Dados/Projetos/UCL_2425/fixturesucl2425.csv'

# Mount Google Drive to save file (if necessary)
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#df.to_csv(file_path, index=False)